<div class="doris-cover">
  <div class="doris-cover-kicker">DEMO 01 · DBT × APACHE DORIS</div>
  <div class="doris-cover-title">每日订单汇总</div>
  <p class="doris-cover-lead">从 6 条订单开始，过滤无效状态，生成每日销售汇总和月度异步物化视图。</p>
  <span class="doris-cover-note">Source · Table · Data Test · Partition · Bucket · Async MV</span>
</div>

## 1. 检查执行环境

先运行下面的单元格。它会读取启动 Jupyter 时传入的 `DBT_BIN` 和 Doris 连接参数，并确认 Backend 可用。

In [ ]:
import importlib.util
from pathlib import Path


def find_demo_dir(start):
    for candidate in (start, *start.parents):
        demo_dir = candidate / "examples/doris-demos"
        if demo_dir.is_dir():
            return demo_dir
    raise FileNotFoundError("请从 dbt-for-apache-doris 仓库目录或其子目录启动 Jupyter。")


demo_dir = find_demo_dir(Path.cwd().resolve())
helper_path = demo_dir / "scripts/notebook_helpers.py"
helper_spec = importlib.util.spec_from_file_location("dbt_doris_notebook_helpers", helper_path)
notebook_helpers = importlib.util.module_from_spec(helper_spec)
helper_spec.loader.exec_module(notebook_helpers)

runner = notebook_helpers.DemoRunner()
runner.show_environment()

## 2. Demo 1：每日订单汇总

这个 Demo 拆成 6 个依次执行的单元格。先运行环境检查，再按 2.1 到 2.6 的顺序执行；每一步都会展示输入或 dbt 文件、运行命令和对应的 Doris 结果。

<div class="doris-flow">
  <div class="doris-flow-step"><strong>源订单</strong>6 条 Doris 订单</div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>过滤</strong>排除取消、退货和失败订单</div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>每日汇总</strong><code>daily_order_summary</code> · 3 行</div><div class="doris-flow-arrow">→</div>
  <div class="doris-flow-step"><strong>月度视图</strong><code>monthly_order_summary_mv</code> · 1 行</div>
</div>

### 2.1 准备并查看源订单

Fixture 创建 6 条订单。最后一列直接标出每条记录会进入模型还是会被过滤。

In [ ]:
daily_demo_dir = runner.examples_root / "doris-daily-order-summary"
runner.show_file("Fixture SQL", daily_demo_dir / "scripts/setup.sql")
runner.run_sql_file("创建源订单", daily_demo_dir / "scripts/setup.sql")
runner.query("输入：6 条原始订单", """
select
    order_id, ordered_at, grand_total, status,
    case
        when status in ('CANCELLED', 'RETURNED', 'FAILED') then '过滤'
        else '进入每日汇总'
    end as transform_action
from dbt_demo_daily_source.orders
order by order_id
""")

### 2.2 将 Doris 表声明为 dbt Source

`sources.yml` 把 `source('orders', 'orders')` 映射到 Doris 的 `dbt_demo_daily_source.orders`。下面用 `dbt ls` 解析 Source 节点，确认项目已经识别到这张源表。连接状态已在环境检查步骤完成。

In [ ]:
runner.show_file("dbt Source 配置", daily_demo_dir / "models/staging/sources.yml")
runner.run_dbt("解析 Demo 的 Source 节点", daily_demo_dir, "ls", "--resource-type", "source")

### 2.3 执行每日汇总 Model

Model 读取 Source，过滤 3 种无效状态，然后按 `order_date` 聚合。dbt-doris 根据 `config()` 创建带 Range Partition、Duplicate Key 和 Hash Bucket 的 Doris Table。

In [ ]:
runner.show_file("每日汇总 Model", daily_demo_dir / "models/marts/daily_order_summary.sql")
runner.run_dbt("创建 daily_order_summary", daily_demo_dir, "run", "--select", "daily_order_summary")
runner.query("输出：3 天的有效订单汇总", """
select order_date, order_count, total_revenue
from dbt_demo_daily.daily_order_summary
order by order_date
""")

### 2.4 执行 Data Test

这一步检查每日结果：日期必须非空且唯一，订单数和收入必须非空。

In [ ]:
runner.show_file("Data Test 定义", daily_demo_dir / "models/marts/daily_order_summary.yml")
runner.run_dbt("验证 daily_order_summary", daily_demo_dir, "test", "--select", "daily_order_summary")

### 2.5 从每日表生成月度异步物化视图

第二个 Model 通过 `ref('daily_order_summary')` 读取上一步的 3 行结果，再按月份聚合。

In [ ]:
runner.show_file("月度物化视图 Model", daily_demo_dir / "models/marts/monthly_order_summary_mv.sql")
runner.run_dbt("创建月度异步物化视图", daily_demo_dir, "run", "--select", "monthly_order_summary_mv")
runner.run_dbt("提交月度物化视图刷新", daily_demo_dir, "run", "--select", "monthly_order_summary_mv")
runner.query("Doris 物化视图任务", """
select MvName, Status, CreateTime
from tasks('type'='mv')
where MvDatabaseName = 'dbt_demo_daily'
  and MvName = 'monthly_order_summary_mv'
order by CreateTime desc
limit 1
""")

### 2.6 校验完整数据链并查看最终结果

Verifier 等待异步刷新完成，同时检查每日数据、Table 的 Key/Partition/Bucket DDL，以及月度汇总值。

In [ ]:
runner.run_script("校验每日订单完整数据链", daily_demo_dir / "scripts/verify.sh")
runner.query("最终结果：月度订单汇总", """
select order_month, order_count, total_revenue
from dbt_demo_daily.monthly_order_summary_mv
order by order_month
""")

## 完成

每日汇总表、四项 Data Test、Doris 表结构和月度异步物化视图均已通过校验。